In [2]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col

In [3]:
url = 'https://github.com/git-iush/large_family/raw/refs/heads/main/data2.parquet'
df = pd.read_parquet(url)

In [4]:
df.head()

,SERIAL,HHWT,CLUSTER,STRATA,PERNUM,PERWT,NCHILD,SEX,AGE,MARST,...,BLACK,AIAN,API,OTHER,MULTR,HISP,EXPANSION,WELFARE,PRSNINC,YRSEDUC
0,4,4.0,2024000000041,280201,1,4.0,0,2,22,6,...,0,0,1,0,0,0,0,0.0,1000.0,13
1,9,74.0,2024000000091,280301,1,74.0,0,2,18,6,...,1,0,0,0,0,0,0,0.0,5100.0,12
2,12,55.0,2024000000121,160001,1,55.0,0,2,19,6,...,0,0,0,0,0,0,0,0.0,0.0,12
3,15,17.0,2024000000151,120201,1,17.0,0,2,19,6,...,0,0,0,0,0,0,0,0.0,6000.0,12
4,19,71.0,2024000000191,10001,1,71.0,0,2,23,6,...,0,0,0,0,0,0,0,0.0,0.0,13


In [5]:
#MEDICAID and HINSCAID  eqaul to eachother
df['MEDICAID'] = df['HINSCAID'] - 1

In [6]:
#age to be quadratic
df['AGE2'] = df['AGE'] ** 2

In [7]:
#Model 1A without the age^2
m1 = smf.ols('NCHILD ~ MEDICAID + WELFARE + AGE' , data = df).fit()

print(m1.summary())

                            OLS Regression Results                            
Dep. Variable:                 NCHILD   R-squared:                       0.199
Model:                            OLS   Adj. R-squared:                  0.199
Method:                 Least Squares   F-statistic:                 5.981e+04
Date:                Sat, 14 Nov 2026   Prob (F-statistic):               0.00
Time:                        21:29:36   Log-Likelihood:            -1.0735e+06
No. Observations:              720668   AIC:                         2.147e+06
Df Residuals:                  720664   BIC:                         2.147e+06
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.8563      0.004   -200.167      0.0

In [8]:
#Age^2 for model 2
m2 = smf.ols('NCHILD ~ MEDICAID + WELFARE + AGE + AGE2' , data = df).fit()

print(m2.summary())

                            OLS Regression Results                            
Dep. Variable:                 NCHILD   R-squared:                       0.225
Model:                            OLS   Adj. R-squared:                  0.225
Method:                 Least Squares   F-statistic:                 5.241e+04
Date:                Sat, 14 Nov 2026   Prob (F-statistic):               0.00
Time:                        21:29:36   Log-Likelihood:            -1.0617e+06
No. Observations:              720668   AIC:                         2.123e+06
Df Residuals:                  720663   BIC:                         2.123e+06
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -2.7525      0.013   -213.306      0.0

In [9]:
#RACE model 3
race = 'BLACK + AIAN + API + OTHER + MULTR + HISP'

m3 = smf.ols(f'NCHILD ~ MEDICAID + WELFARE + AGE + AGE2 + {race}' , data = df).fit()

print(m3.summary())

                            OLS Regression Results                            
Dep. Variable:                 NCHILD   R-squared:                       0.228
Model:                            OLS   Adj. R-squared:                  0.228
Method:                 Least Squares   F-statistic:                 2.133e+04
Date:                Sat, 14 Nov 2026   Prob (F-statistic):               0.00
Time:                        21:29:37   Log-Likelihood:            -1.0602e+06
No. Observations:              720668   AIC:                         2.120e+06
Df Residuals:                  720657   BIC:                         2.121e+06
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -2.7514      0.013   -212.631      0.0

In [10]:
#Compare
table = summary_col([m1, m2, m3],
                    model_names = ['(1) Age' , '(2) + age^2' , '(3) + Race'],
                    regressor_order = ['MEDICAID' , 'WELFARE' , 'AGE' , 'AGE2'])

print(table)


               (1) Age  (2) + age^2 (3) + Race
----------------------------------------------
MEDICAID       0.2074   0.2274      0.2219    
               (0.0033) (0.0033)    (0.0033)  
WELFARE        0.1264   0.0520      0.0518    
               (0.0099) (0.0098)    (0.0098)  
AGE            0.0516   0.1818      0.1819    
               (0.0001) (0.0008)    (0.0008)  
AGE2                    -0.0020     -0.0020   
                        (0.0000)    (0.0000)  
Intercept      -0.8563  -2.7525     -2.7514   
               (0.0043) (0.0129)    (0.0129)  
BLACK                               -0.1098   
                                    (0.0045)  
AIAN                                0.0918    
                                    (0.0105)  
API                                 -0.1381   
                                    (0.0047)  
OTHER                               0.0138    
                                    (0.0062)  
MULTR                               -0.0443   
            

MEDICAID
0    583555
1    137113
Name: count, dtype: int64